# 1차 PatchCore 타당성 확인 (Feasibility Check)

`report/model_selection_report.md` 6.1절의 1차 실험 노트북입니다.
**정상 이미지만으로 학습한 PatchCore가 정상과 불량을 구분할 수 있는지** 확인합니다.

| 단계 | 내용 |
| :--- | :--- |
| 0 | Drive 마운트, 라이브러리 설치, 실험 설정 |
| 1 | 데이터 정리: 태그 → 제외(도금·녹, 완전 중복) → 그룹 단위 분할 (`split.csv`) |
| 2 | 전처리 (B1~B7) → 256×256 크롭 |
| 3 | 학습: 정상 이미지로 정상 패치 메모리 생성 |
| 4 | 추론: 검증·평가 세트 이상 점수와 히트맵 |
| 5 | 임계값 결정 (미검 우선 / F1 최대) |
| 6 | 평가: AUROC, 미검률·과검률(95% 신뢰구간), F1, 그룹별 성능, 속도·크기 |
| 7 | 시각화 |
| 8 | 결과 저장 (지표, 히트맵, YOLO 박스 초안) |

**실험 바꾸기**: 0단계 설정 셀의 `EXP_NAME`을 `P1_baseline` / `P2_blur_aug` / `P3_light` 중 하나로 바꾸고 `런타임 → 모두 실행`.
모든 실험은 같은 `split.csv`를 쓰므로 결과를 바로 비교할 수 있습니다.

> GPU 런타임 권장 (`런타임 → 런타임 유형 변경 → GPU`). CPU에서도 돌지만 느립니다.

## 0. 준비

In [ ]:
# Google Drive 마운트 (로컬 Jupyter에서 실행하면 건너뜀)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Colab:", IN_COLAB)

In [ ]:
# anomalib 설치 — 버전을 고정해야 나중에 다시 돌려도 같은 결과가 나옴.
# torch는 Colab에 이미 설치된 것을 그대로 사용 (extra 옵션 없이 설치).
# 설치 중 의존성 경고가 나와도 아래 import 셀이 통과하면 문제없음.
if IN_COLAB:
    !pip install -q anomalib==2.7.0

In [ ]:
import os, re, glob, json, time, hashlib, shutil, random
from collections import Counter, defaultdict

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.metrics import roc_auc_score, roc_curve

import anomalib
# Lightning 래퍼(Patchcore + Engine) 대신 내부 torch 모델을 직접 사용.
# 이유: 래퍼는 점수를 0~1로 자동 정규화하고 임계값도 자동으로 정하는데,
#       이 실험은 원시 점수로 임계값을 직접 정하고 지표도 직접 계산해야 하기 때문.
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("anomalib", anomalib.__version__, "| torch", torch.__version__, "| device", DEVICE)

In [ ]:
# ---------------------------------------------------------------
# 경로 설정 — 데이터셋을 업로드한 위치에 맞게 수정
# ---------------------------------------------------------------
if IN_COLAB:
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/industrial-nut-defect-synthetic-dataset"
    WORK_DIR = "/content/drive/MyDrive/patchcore_out"   # split.csv와 결과가 남는 곳 (Drive라서 런타임이 끊겨도 유지)
    DATA_ROOT = "/content/dataset"                      # Drive → 로컬 디스크로 복사 (반복 읽기 속도 향상)
    if not os.path.isdir(DATA_ROOT):
        shutil.copytree(DRIVE_DATA_ROOT, DATA_ROOT)
else:
    DATA_ROOT = "../industrial-nut-defect-synthetic-dataset"
    WORK_DIR = "patchcore_out"

for sub in ["synthetic_defect", "synthetic_non_defect"]:
    assert os.path.isdir(os.path.join(DATA_ROOT, sub)), f"폴더 없음: {DATA_ROOT}/{sub}"
os.makedirs(WORK_DIR, exist_ok=True)

# ---------------------------------------------------------------
# 실험 설정 — 여기만 바꿔서 P-1 / P-2 / P-3 실행
# ---------------------------------------------------------------
EXP_NAME = "P1_baseline"

EXPERIMENTS = {
    # P-1 기본: 보고서 기본 구성 (WideResNet-50, layer2+3, 코어셋 10%)
    "P1_baseline": dict(backbone="wide_resnet50_2", layers=["layer2", "layer3"],
                        coreset_ratio=0.10, num_neighbors=9, blur_aug=False, fp16=False),
    # P-2 흐림 대응: 학습용 정상 이미지에 '축소→재확대' 열화본을 추가해 흐린 정상도 정상으로 기억하게 함
    "P2_blur_aug": dict(backbone="wide_resnet50_2", layers=["layer2", "layer3"],
                        coreset_ratio=0.10, num_neighbors=9, blur_aug=True, fp16=False),
    # P-3 경량화: 작은 백본 + 작은 메모리 + FP16 (GPU에서만 FP16 적용)
    "P3_light":    dict(backbone="resnet18", layers=["layer2", "layer3"],
                        coreset_ratio=0.05, num_neighbors=9, blur_aug=False, fp16=True),
}
CFG = EXPERIMENTS[EXP_NAME]
SEED = 42            # 분할·코어셋 샘플링 재현용
IMG_SIZE = 256       # 전처리 출력 = 모델 입력 크기
BATCH = 16

OUT_DIR = os.path.join(WORK_DIR, EXP_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(EXP_NAME, CFG)
print("DATA_ROOT:", DATA_ROOT, "| OUT_DIR:", OUT_DIR)

## 1. 데이터 정리

`data_preprocessing.md` A1~A2, A6을 그대로 구현합니다.
1. 이미지별 태그: 품종(`material`), 흐림(`blur`), 원거리(`scale`), 중복 그룹(`group_id`)
2. 제외: 도금·녹 너트 (정상 데이터가 없음), 완전 중복 (1장만 남김)
3. **그룹 단위 분할**: 유사 이미지가 학습과 평가에 나뉘어 성능이 부풀려지는 것을 막음

`split.csv`가 이미 있으면 **다시 만들지 않고 불러옵니다.** 그래야 P-1~P-3, 그리고 2차 YOLO까지 같은 평가 세트를 씁니다.

In [ ]:
CLASSES = {"synthetic_defect": 1, "synthetic_non_defect": 0}   # 1 = 불량, 0 = 정상

# 육안 분류 결과 (근사치, 검수 필요) — 불량 폴더의 파일 번호
ZINC_IDS = {2, 13, 16, 18, 27, 38, 44, 45, 49, 60, 64, 72, 79, 80, 88, 93, 104, 105, 110, 122, 127, 130}
RUST_IDS = {3, 5, 17, 23, 30, 46, 53, 56, 59, 63, 69, 81, 90, 112, 115, 117, 119, 126, 128}

BLUR_LAPVAR = 30.0   # 라플라시안 분산이 이보다 작으면 '흐림'
SMALL_AREA = 0.12    # 부품이 화면에서 차지하는 비율이 이보다 작으면 '원거리'
PHASH_DIST = 6       # 지각 해시 거리 이하이면 '유사 이미지'


def list_images(root):
    items = []
    for cls, label in CLASSES.items():
        for f in glob.glob(os.path.join(root, cls, "*.png")):
            idx = int(re.findall(r"(\d+)\.png$", f)[0])
            items.append({"path": f, "file": os.path.basename(f), "cls": cls, "label": label, "idx": idx})
    return sorted(items, key=lambda it: (it["cls"], it["idx"]))


def border_pixels(img, width=20):
    # 화면 테두리 띠의 픽셀 — 배경 밝기 추정에 사용
    return np.concatenate([img[:width].ravel(), img[-width:].ravel(),
                           img[:, :width].ravel(), img[:, -width:].ravel()])


def phash(gray, hash_size=8):
    # DCT 기반 지각 해시: 이미지를 32x32로 줄여 저주파 성분의 대소만 64비트로 기록
    small = cv2.resize(gray, (hash_size * 4, hash_size * 4), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(small)[:hash_size, :hash_size]
    return (dct > np.median(dct)).ravel()


items = list_images(DATA_ROOT)
for it in items:
    bgr = cv2.imread(it["path"])
    g = bgr.astype(np.float32).mean(axis=2)                      # 채널 평균 흑백
    bg = border_pixels(g).mean()
    it["md5"] = hashlib.md5(open(it["path"], "rb").read()).hexdigest()
    it["phash"] = phash(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))
    it["lapvar"] = float(cv2.Laplacian(g, cv2.CV_32F).var())    # 선명도
    it["area"] = float((np.abs(g - bg) > 35).mean())             # 배경과 다른 픽셀 비율 ≈ 부품 면적
    it["blur"] = "blurry" if it["lapvar"] < BLUR_LAPVAR else "sharp"
    it["scale"] = "small" if it["area"] < SMALL_AREA else "normal"
    if it["label"] == 1 and it["idx"] in ZINC_IDS:
        it["material"] = "zinc"
    elif it["label"] == 1 and it["idx"] in RUST_IDS:
        it["material"] = "rust"
    else:
        it["material"] = "plain"

print(f"전체 {len(items)}장:", dict(Counter((it['cls'], it['material']) for it in items)))

In [ ]:
# ---- 중복 그룹: 같은 클래스 안에서 pHash 거리가 가까운 이미지를 Union-Find로 묶음 ----
parent = {it["file"]: it["file"] for it in items}

def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

for cls in CLASSES:
    sub = [it for it in items if it["cls"] == cls]
    for i, a in enumerate(sub):
        for b in sub[i + 1:]:
            if np.count_nonzero(a["phash"] != b["phash"]) <= PHASH_DIST:
                parent[find(a["file"])] = find(b["file"])
for it in items:
    it["group"] = find(it["file"])

# ---- 제외: 도금·녹 너트 + 완전 중복(같은 MD5는 첫 장만 남김) ----
seen_md5, data = set(), []
for it in items:
    if it["material"] != "plain":
        continue
    if it["md5"] in seen_md5:
        continue
    seen_md5.add(it["md5"])
    data.append(it)

n_groups = {c: len({it['group'] for it in data if it['cls'] == c}) for c in CLASSES}
print(f"1차 범위 {len(data)}장:", dict(Counter(it['cls'] for it in data)), "| 고유 그룹:", n_groups)

In [ ]:
# ---- 그룹 단위 층화 분할 (70 / 15 / 15) ----
# 같은 그룹은 반드시 같은 세트로 가고, (클래스, 흐림, 원거리) 조합별로 비율을 맞춤.
SPLIT_CSV = os.path.join(WORK_DIR, "split.csv")

def make_split(data, seed=SEED, ratios=(0.70, 0.15, 0.15)):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for it in data:
        groups[it["group"]].append(it)
    # 그룹의 층(stratum) = 그룹 대표 이미지의 (클래스, 흐림, 원거리)
    strata = defaultdict(list)
    for g, members in groups.items():
        rep = members[0]
        strata[(rep["label"], rep["blur"], rep["scale"])].append(g)
    assign = {}
    for key in sorted(strata, key=str):
        gs = sorted(strata[key]); rng.shuffle(gs)
        n = len(gs)
        n_val = int(round(n * ratios[1])); n_test = int(round(n * ratios[2]))
        n_train = max(n - n_val - n_test, 0)
        for i, g in enumerate(gs):
            assign[g] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
    return {it["file"]: assign[it["group"]] for it in data}

if os.path.exists(SPLIT_CSV):
    # 이미 만든 분할을 재사용 → 실험 간 평가 세트 고정
    split = {}
    for line in open(SPLIT_CSV, encoding="utf-8").read().splitlines()[1:]:
        f, s = line.split(",")[:2]
        split[f] = s
    print("기존 split.csv 사용:", SPLIT_CSV)
else:
    split = make_split(data)
    with open(SPLIT_CSV, "w", encoding="utf-8") as fp:
        fp.write("file,split,label,material,blur,scale,group\n")
        for it in data:
            fp.write(f"{it['file']},{split[it['file']]},{it['label']},{it['material']},{it['blur']},{it['scale']},{it['group']}\n")
    print("새 split.csv 저장:", SPLIT_CSV)

for it in data:
    it["split"] = split[it["file"]]

print("세트별 장수 (정상 / 불량):")
for s in ["train", "val", "test"]:
    sub = [it for it in data if it["split"] == s]
    print(f"  {s:5s} 정상 {sum(it['label'] == 0 for it in sub):3d} / 불량 {sum(it['label'] == 1 for it in sub):3d}"
          f"  | 흐림 {sum(it['blur'] == 'blurry' for it in sub)}장, 원거리 {sum(it['scale'] == 'small' for it in sub)}장")

## 2. 전처리 (B1~B7)

`data_preprocessing.md` 3장의 엣지 전처리를 그대로 적용해 256×256 크롭을 만듭니다.
B8(정규화)은 백본이 ImageNet으로 학습되었으므로 ImageNet 평균·표준편차로 대신합니다.

추가로 **회전·크롭 변환 행렬(M)** 을 저장해 둡니다. 히트맵에서 찾은 결함 위치를 원본 좌표로 되돌려, 2차 YOLO 라벨링 초안으로 쓰기 위해서입니다.

In [ ]:
BG_TARGET = 220      # 배경 밝기 정규화 목표값
CROP_MARGIN = 0.10   # 부품 박스 주변 여백 (사방 10%)


def roi_gradient(gray):
    # B3 ROI 검출: 밝기 변화(그래디언트)가 큰 곳 = 부품 외곽.
    # 금속 면이 배경만큼 밝아도 경계는 찾을 수 있어서 Otsu 이진화보다 안정적.
    h, w = gray.shape
    s = cv2.resize(gray, (w // 2, h // 2), interpolation=cv2.INTER_AREA)     # 1/2 축소로 속도 확보
    b = cv2.GaussianBlur(s, (5, 5), 0)
    mag = cv2.magnitude(cv2.Sobel(b, cv2.CV_32F, 1, 0), cv2.Sobel(b, cv2.CV_32F, 0, 1))
    # 배경은 매끈하므로 테두리 그래디언트의 중앙값 ≈ 배경 잡음 수준 → 그 4배를 임계값으로
    t = float(np.clip(np.median(border_pixels(mag, 8)) * 4, 12, 60))
    m = (mag > t).astype(np.uint8)
    # 엣지를 팽창시켜 이어 붙인 뒤 가장 큰 덩어리만 부품으로 채택 (배경 잡티 제거)
    n, lab, st, _ = cv2.connectedComponentsWithStats(cv2.dilate(m, np.ones((9, 9), np.uint8)))
    if n <= 1:
        return None
    biggest = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    pts = cv2.findNonZero(((lab == biggest) & (m > 0)).astype(np.uint8))
    if pts is None or len(pts) < 30:
        return None
    return cv2.convexHull(pts) * 2   # 원본 좌표로 복원


def preprocess(bgr, out_size=IMG_SIZE):
    # 반환: (256x256 흑백 크롭, 원본→크롭 변환 행렬 M, ROI 실패 여부)
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)                                 # B1 흑백
    gain = BG_TARGET / max(float(np.median(border_pixels(g))), 1.0)
    n = cv2.convertScaleAbs(g, alpha=gain, beta=0)                            # B2 배경 밝기 정규화
    hull = roi_gradient(g)                                                    # B3 ROI
    H, W = g.shape
    ratio = 0 if hull is None else cv2.contourArea(hull) / (H * W)
    fail = hull is None or ratio > 0.90 or ratio < 0.08
    (cx, cy), (rw, rh), angle = ((W / 2, H / 2), (W, H), 0.0) if fail else cv2.minAreaRect(hull)
    # B4~B6: 회전 정렬 + 크롭 + 크기 정규화 + 배경색 패딩을 warpAffine 한 번으로 처리
    side = max(rw, rh) * (1 + 2 * CROP_MARGIN)
    M = cv2.getRotationMatrix2D((cx, cy), angle, out_size / max(side, 1.0))
    M[0, 2] += out_size / 2 - cx
    M[1, 2] += out_size / 2 - cy
    a = cv2.warpAffine(n, M, (out_size, out_size), flags=cv2.INTER_LINEAR, borderValue=BG_TARGET)
    c = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8)).apply(a)          # B7 약한 CLAHE
    return c, M, fail


def degrade(bgr, rng):
    # P-2용 열화: 축소 후 재확대 → 데이터셋의 '업스케일 흐림'을 흉내냄
    f = rng.uniform(0.3, 0.6)
    h, w = bgr.shape[:2]
    small = cv2.resize(bgr, (max(int(w * f), 1), max(int(h * f), 1)), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


t0 = time.perf_counter()
for it in data:
    it["crop"], it["M"], it["roi_fail"] = preprocess(cv2.imread(it["path"]))
print(f"전처리 {len(data)}장, {time.perf_counter() - t0:.1f}s | ROI 실패 {sum(it['roi_fail'] for it in data)}장")

# 미리보기: 세트별 정상·불량 몇 장
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for r, lab in enumerate([0, 1]):
    sub = [it for it in data if it["label"] == lab][:6]
    for c, it in enumerate(sub):
        axes[r, c].imshow(it["crop"], cmap="gray", vmin=0, vmax=255)
        axes[r, c].set_title(f"{'normal' if lab == 0 else 'defect'} | {it['file'][6:-4]}", fontsize=8)
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 흑백 크롭 → 모델 입력 텐서 (N, 3, 256, 256)
# 백본이 RGB ImageNet으로 학습되었으므로 흑백을 3채널로 복제하고 ImageNet 기준으로 정규화.
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def to_tensor(crops):
    x = torch.from_numpy(np.stack(crops)).float().div(255.0)      # (N, H, W), 0~1
    x = x.unsqueeze(1).repeat(1, 3, 1, 1)                          # (N, 3, H, W)
    return (x - MEAN) / STD

train_normals = [it for it in data if it["split"] == "train" and it["label"] == 0]
train_crops = [it["crop"] for it in train_normals]

if CFG["blur_aug"]:
    # P-2: 학습 정상 이미지마다 열화본 1장씩 추가 → 메모리에 '흐린 정상'도 들어감
    rng = random.Random(SEED)
    train_crops += [preprocess(degrade(cv2.imread(it["path"]), rng))[0] for it in train_normals]

print(f"학습 입력: 정상 {len(train_normals)}장" + (f" + 열화본 {len(train_normals)}장" if CFG["blur_aug"] else ""))

## 3. 학습 — 정상 패치 메모리 만들기

PatchCore의 "학습"은 역전파가 아니라 다음 두 단계입니다.
1. 정상 이미지를 백본에 통과시켜 **패치별 특징 벡터**를 모읍니다 (layer2 + layer3 특징을 합쳐 사용).
2. **코어셋 샘플링**으로 그중 대표 벡터만 남겨 메모리 크기를 줄입니다 (`coreset_ratio`).

그래서 에폭·학습률 같은 설정이 없고, 정상 이미지를 한 번 통과시키면 끝납니다.

In [ ]:
use_fp16 = CFG["fp16"] and DEVICE == "cuda"     # CPU는 FP16 합성곱을 지원하지 않으므로 GPU에서만

model = PatchcoreModel(
    layers=list(CFG["layers"]),          # 특징을 뽑을 백본 레이어 (anomalib 내부에서 수정하므로 list로 전달)
    backbone=CFG["backbone"],            # timm 백본 이름 — ImageNet 사전 학습 가중치를 자동 다운로드
    pre_trained=True,
    num_neighbors=CFG["num_neighbors"],  # 이미지 점수 보정에 쓰는 이웃 수
).to(DEVICE)

# model.train()은 '특징을 메모리에 쌓는 모드'로 바꾸는 스위치.
# 단, 백본의 BatchNorm까지 학습 모드가 되면 특징이 배치마다 달라지므로 백본은 eval로 고정.
model.train()
model.feature_extractor.eval()

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

t0 = time.perf_counter()
with torch.no_grad():
    x_all = to_tensor(train_crops)
    for i in range(0, len(x_all), BATCH):
        model(x_all[i:i + BATCH].to(DEVICE))       # 학습 모드: 패치 특징을 embedding_store에 누적
    n_patches = sum(e.shape[0] for e in model.embedding_store)
    model.subsample_embedding(CFG["coreset_ratio"])  # 코어셋 샘플링 → memory_bank 생성
sync()
TRAIN_SEC = time.perf_counter() - t0

if use_fp16:
    model = model.half()                 # 추론용 FP16 변환 (메모리 뱅크도 함께 변환됨)
model.eval()

print(f"학습 시간 {TRAIN_SEC:.1f}s | 수집 패치 {n_patches:,}개 → 메모리 {model.memory_bank.shape[0]:,}개 "
      f"(차원 {model.memory_bank.shape[1]}) | FP16={use_fp16}")

## 4. 추론 — 이상 점수와 히트맵

각 패치가 메모리 속 가장 가까운 정상 패치와 얼마나 먼지 계산합니다.
- **이상 점수** (`pred_score`): 가장 먼 패치의 거리 (이웃 정보로 보정). 클수록 불량에 가까움.
- **히트맵** (`anomaly_map`): 패치별 거리를 이미지 크기로 펼친 것. 결함 위치를 보여 줌.

학습에 안 쓴 **모든 이미지**(검증·평가 세트 + 학습 세트의 불량)에 대해 계산합니다. 학습 세트 불량의 히트맵은 2차 YOLO 라벨링 초안에 씁니다.

In [ ]:
def infer(crops):
    scores, maps = [], []
    with torch.no_grad():
        x_all = to_tensor(crops)
        for i in range(0, len(x_all), BATCH):
            x = x_all[i:i + BATCH].to(DEVICE)
            out = model(x.half() if use_fp16 else x)                   # 추론 모드: InferenceBatch 반환
            scores.append(out.pred_score.float().reshape(-1).cpu().numpy())
            maps.append(out.anomaly_map.float().squeeze(1).cpu().numpy())
    return np.concatenate(scores), np.concatenate(maps)

targets = [it for it in data if not (it["split"] == "train" and it["label"] == 0)]
scores, maps = infer([it["crop"] for it in targets])
for it, s, m in zip(targets, scores, maps):
    it["score"], it["amap"] = float(s), m

val = [it for it in data if it["split"] == "val"]
test = [it for it in data if it["split"] == "test"]
print(f"추론 완료: {len(targets)}장 | 검증 {len(val)}장, 평가 {len(test)}장")

In [ ]:
# ---- 이미지 1장당 지연시간 (전처리 + 추론, 배치 크기 1) ----
# 엣지에서는 한 장씩 들어오므로 배치 1로 측정. 첫 몇 번은 GPU 워밍업이라 제외.
lat_pre, lat_inf = [], []
sample = test[:30]
with torch.no_grad():
    for k, it in enumerate([test[0]] * 5 + sample):
        bgr = cv2.imread(it["path"])
        t0 = time.perf_counter()
        crop = preprocess(bgr)[0]
        t1 = time.perf_counter()
        x = to_tensor([crop]).to(DEVICE)
        model(x.half() if use_fp16 else x)
        sync()
        t2 = time.perf_counter()
        if k >= 5:
            lat_pre.append((t1 - t0) * 1000); lat_inf.append((t2 - t1) * 1000)

# ---- 모델 크기: 저장 파일 기준 (백본 가중치 + 메모리 뱅크) ----
MODEL_PATH = os.path.join(OUT_DIR, "patchcore.pt")
torch.save(model.state_dict(), MODEL_PATH)
MODEL_MB = os.path.getsize(MODEL_PATH) / 1e6
BANK_MB = model.memory_bank.numel() * model.memory_bank.element_size() / 1e6

print(f"지연시간(median) 전처리 {np.median(lat_pre):.2f}ms + 추론 {np.median(lat_inf):.2f}ms "
      f"= {np.median(np.add(lat_pre, lat_inf)):.2f}ms  (p95 {np.percentile(np.add(lat_pre, lat_inf), 95):.2f}ms, {DEVICE})")
print(f"모델 파일 {MODEL_MB:.1f}MB (그중 메모리 뱅크 {BANK_MB:.1f}MB)")

## 5. 임계값 결정 (검증 세트)

이상 점수가 임계값 **이상이면 불량**으로 판정합니다. 임계값은 검증 세트로만 정합니다 (평가 세트는 건드리지 않음).

- **① 미검 우선 (기본)**: 검증 세트의 불량을 **하나도 놓치지 않는** 가장 높은 임계값 = 검증 불량 점수의 최솟값. PRD가 미검률(0.01%)을 과검률(0.5%)보다 훨씬 엄격하게 요구하기 때문.
- **② F1 최대**: 정상·불량 판정의 균형이 가장 좋은 임계값. ①과 비교용.

In [ ]:
v_scores = np.array([it["score"] for it in val])
v_labels = np.array([it["label"] for it in val])

# ① 미검 우선: 검증 불량 중 가장 낮은 점수 → 이 값 이상이면 검증 불량 전부 검출
THR_RECALL = float(v_scores[v_labels == 1].min())

# ② F1 최대: 검증 점수들을 후보로 두고 F1이 가장 높은 값 선택
def f1_at(thr, s, y):
    pred = s >= thr
    tp = np.sum(pred & (y == 1)); fp = np.sum(pred & (y == 0)); fn = np.sum(~pred & (y == 1))
    return 2 * tp / max(2 * tp + fp + fn, 1)

cands = np.unique(v_scores)
THR_F1 = float(max(cands, key=lambda t: f1_at(t, v_scores, v_labels)))

THRESHOLDS = {"recall_first": THR_RECALL, "f1_max": THR_F1}
print("임계값:", {k: round(v, 4) for k, v in THRESHOLDS.items()})
print(f"검증 세트 AUROC {roc_auc_score(v_labels, v_scores):.3f}")

## 6. 평가 (평가 세트)

- **AUROC**: 임계값과 무관하게 정상·불량 점수가 얼마나 잘 갈리는지 (1.0 = 완벽 분리, 0.5 = 무작위)
- **미검률(FNR)** = 놓친 불량 / 전체 불량, **과검률(FPR)** = 불량으로 잘못 판정한 정상 / 전체 정상
- **95% 신뢰구간 (Clopper-Pearson)**: 표본이 적어서 점추정만으로는 의미가 약함. 예를 들어 불량 13장 중 0장을 놓쳐도 미검률 상한은 약 25%.
- **그룹별 성능**: 흐림/선명, 원거리/근거리로 나눠 봄. 그룹 간 차이가 크면 모델이 결함이 아니라 화질·크기로 판단하고 있다는 신호.

In [ ]:
def clopper_pearson(k, n, alpha=0.05):
    # k/n 비율의 정확한 95% 신뢰구간
    if n == 0:
        return (float("nan"), float("nan"))
    lo = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)
    return (float(lo), float(hi))


def evaluate(sub, thr):
    s = np.array([it["score"] for it in sub]); y = np.array([it["label"] for it in sub])
    pred = s >= thr
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    fn = int((~pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum()); tp = n_pos - fn
    auroc = roc_auc_score(y, s) if n_pos and n_neg else float("nan")
    return {
        "n_normal": n_neg, "n_defect": n_pos, "auroc": auroc,
        "fn": fn, "fnr": fn / n_pos if n_pos else float("nan"), "fnr_ci": clopper_pearson(fn, n_pos),
        "fp": fp, "fpr": fp / n_neg if n_neg else float("nan"), "fpr_ci": clopper_pearson(fp, n_neg),
        "f1": 2 * tp / max(2 * tp + fp + fn, 1),
    }


def fmt(r):
    return (f"AUROC {r['auroc']:.3f} | FNR {r['fn']}/{r['n_defect']} = {r['fnr']:.1%} "
            f"(95% CI {r['fnr_ci'][0]:.1%}~{r['fnr_ci'][1]:.1%}) | "
            f"FPR {r['fp']}/{r['n_normal']} = {r['fpr']:.1%} (95% CI {r['fpr_ci'][0]:.1%}~{r['fpr_ci'][1]:.1%}) | F1 {r['f1']:.3f}")


RESULTS = {}
for name, thr in THRESHOLDS.items():
    RESULTS[name] = evaluate(test, thr)
    print(f"[평가 · {name} 임계값 {thr:.4f}]\n  {fmt(RESULTS[name])}")

In [ ]:
# ---- 그룹별 성능: 흐림 / 원거리 ----
# AUROC는 그룹 안에 정상·불량이 모두 있어야 계산됨 (없으면 nan)
STRATA = {}
print("그룹별 성능 (미검 우선 임계값 기준)")
for key in ["blur", "scale"]:
    for val_ in sorted({it[key] for it in test}):
        sub = [it for it in test if it[key] == val_]
        r = evaluate(sub, THRESHOLDS["recall_first"])
        STRATA[f"{key}={val_}"] = r
        print(f"  {key}={val_:7s} 정상 {r['n_normal']:2d} / 불량 {r['n_defect']:2d} | {fmt(r)}")

a_sharp = STRATA.get("blur=sharp", {}).get("auroc", float("nan"))
a_blur = STRATA.get("blur=blurry", {}).get("auroc", float("nan"))
BLUR_GAP = abs(a_sharp - a_blur)
print(f"\n흐림/선명 AUROC 차이: {BLUR_GAP:.3f}  (통과 기준 ≤ 0.10, nan이면 한쪽 그룹에 정상 또는 불량이 없는 것)")

# 정상 이미지끼리만 비교: 흐린 정상의 점수가 선명한 정상보다 높으면 '흐림 = 이상'으로 보고 있다는 뜻
ns = [it for it in test + val if it["label"] == 0]
for b in ["sharp", "blurry"]:
    v = [it["score"] for it in ns if it["blur"] == b]
    if v:
        print(f"  정상({b}) 점수 평균 {np.mean(v):.3f} (n={len(v)})")

## 7. 시각화

In [ ]:
t_scores = np.array([it["score"] for it in test]); t_labels = np.array([it["label"] for it in test])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# (1) 점수 분포: 두 분포가 겹치지 않을수록 좋음
bins = np.linspace(min(t_scores.min(), v_scores.min()), max(t_scores.max(), v_scores.max()), 30)
axes[0].hist(t_scores[t_labels == 0], bins=bins, alpha=0.6, color="tab:blue", label="normal (test)")
axes[0].hist(t_scores[t_labels == 1], bins=bins, alpha=0.6, color="tab:red", label="defect (test)")
for (name, thr), ls in zip(THRESHOLDS.items(), ["--", ":"]):
    axes[0].axvline(thr, color="k", ls=ls, label=f"thr {name}")
axes[0].set_xlabel("anomaly score"); axes[0].set_title(f"{EXP_NAME} score distribution"); axes[0].legend(fontsize=8)
# (2) ROC 곡선
fpr_, tpr_, _ = roc_curve(t_labels, t_scores)
axes[1].plot(fpr_, tpr_, color="tab:purple"); axes[1].plot([0, 1], [0, 1], "k:", lw=0.8)
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title(f"ROC (test AUROC {RESULTS['recall_first']['auroc']:.3f})")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "scores_roc.png"), dpi=100); plt.show()

In [ ]:
def show_heatmaps(sub, title, n=8, path=None):
    # 위: 전처리 크롭 / 아래: 히트맵 오버레이 (빨간 곳 = 정상과 많이 다른 곳)
    sub = sub[:n]
    if not sub:
        print(f"{title}: 해당 없음"); return
    vmax = max(it["amap"].max() for it in sub)
    fig, axes = plt.subplots(2, len(sub), figsize=(2.3 * len(sub), 5), squeeze=False)
    for j, it in enumerate(sub):
        axes[0, j].imshow(it["crop"], cmap="gray", vmin=0, vmax=255)
        axes[0, j].set_title(f"{it['file'][6:-4]}\nscore {it['score']:.2f}", fontsize=8)
        axes[1, j].imshow(it["crop"], cmap="gray", vmin=0, vmax=255)
        axes[1, j].imshow(it["amap"], cmap="jet", alpha=0.45, vmin=0, vmax=vmax)
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout()
    if path:
        plt.savefig(path, dpi=90)
    plt.show()

thr = THRESHOLDS["recall_first"]
t_def = sorted([it for it in test if it["label"] == 1], key=lambda it: -it["score"])
show_heatmaps(t_def, "test defects (highest score first)", path=os.path.join(OUT_DIR, "heatmaps_test_defects.png"))
show_heatmaps([it for it in t_def if it["score"] < thr], "MISSED defects (FN)")
show_heatmaps(sorted([it for it in test if it["label"] == 0 and it["score"] >= thr], key=lambda it: -it["score"]),
              "FALSE alarms on normals (FP)")

## 8. 결과 저장

- `OUT_DIR/metrics.json`: 이 실험의 전체 지표
- `WORK_DIR/results_all.csv`: 실험마다 한 줄씩 누적 → 보고서 6.4절 결과표에 그대로 옮기면 됨
- `OUT_DIR/yolo_drafts/`: 불량 이미지별 히트맵과 **박스 초안** (2차 YOLO 라벨링 시작점)

In [ ]:
r = RESULTS["recall_first"]
summary = {
    "exp": EXP_NAME, **{k: (list(v) if isinstance(v, list) else v) for k, v in CFG.items()},
    "anomalib": anomalib.__version__, "device": DEVICE,
    "n_train_normal": len(train_normals), "memory_bank": int(model.memory_bank.shape[0]),
    "thresholds": THRESHOLDS, "test": RESULTS, "strata": STRATA, "blur_auroc_gap": BLUR_GAP,
    "train_sec": TRAIN_SEC, "latency_ms_median": float(np.median(np.add(lat_pre, lat_inf))),
    "latency_ms_p95": float(np.percentile(np.add(lat_pre, lat_inf), 95)), "model_mb": MODEL_MB,
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=float)

# 보고서 6.4절 결과표 형식으로 한 줄 추가 (같은 실험을 다시 돌리면 해당 줄을 덮어씀)
ALL_CSV = os.path.join(WORK_DIR, "results_all.csv")
header = "exp,auroc,fnr,fnr_ci_hi,fpr,fpr_ci_hi,f1,blur_auroc_gap,train_sec,latency_ms,model_mb,device"
row = (f"{EXP_NAME},{r['auroc']:.4f},{r['fnr']:.4f},{r['fnr_ci'][1]:.4f},{r['fpr']:.4f},{r['fpr_ci'][1]:.4f},"
       f"{r['f1']:.4f},{BLUR_GAP:.4f},{TRAIN_SEC:.1f},{summary['latency_ms_median']:.2f},{MODEL_MB:.1f},{DEVICE}")
lines = open(ALL_CSV, encoding="utf-8").read().splitlines()[1:] if os.path.exists(ALL_CSV) else []
lines = [l for l in lines if not l.startswith(EXP_NAME + ",")] + [row]
with open(ALL_CSV, "w", encoding="utf-8") as fp:
    fp.write("\n".join([header] + lines) + "\n")
print(open(ALL_CSV, encoding="utf-8").read())

In [ ]:
# ---- 2차 YOLO 라벨링 초안: 히트맵 → 박스 → 원본 좌표 ----
# 1) 회전·크롭 때 생긴 패딩 영역은 원본에 없던 부분이므로 히트맵에서 제외
#    (패딩 경계선이 이상 부위로 잡히는 경우가 있음)
# 2) 히트맵 최고점을 포함하고, 값이 (최소 + 80% x 범위) 이상인 덩어리를 결함 후보로 보고 박스를 침
#    → 60%로 낮추면 큰 너트에서 박스가 너트 전체만큼 커지는 것을 확인해 80%로 설정
# 3) 전처리 때 저장한 변환 행렬 M의 역변환으로 박스 네 꼭짓점을 원본 640x640 좌표로 되돌림
# 4) YOLO 형식(클래스 중심x 중심y 너비 높이, 0~1 정규화)으로 저장 → 라벨링 도구에서 불러와 사람이 수정
DRAFT_DIR = os.path.join(OUT_DIR, "yolo_drafts")
os.makedirs(os.path.join(DRAFT_DIR, "labels"), exist_ok=True)
os.makedirs(os.path.join(DRAFT_DIR, "overlays"), exist_ok=True)

defects = [it for it in data if it["label"] == 1]
for it in defects:
    H, W = cv2.imread(it["path"], cv2.IMREAD_GRAYSCALE).shape
    # 원본이 있던 영역 마스크: 흰 이미지를 같은 변환으로 옮기면 패딩 부분만 0이 됨
    valid = cv2.warpAffine(np.full((H, W), 255, np.uint8), it["M"], (IMG_SIZE, IMG_SIZE), borderValue=0)
    valid = cv2.erode(valid, np.ones((9, 9), np.uint8)) > 0
    amap = np.where(valid, it["amap"], it["amap"].min())
    peak = np.unravel_index(np.argmax(amap), amap.shape)
    mask = (amap >= amap.min() + 0.8 * (amap.max() - amap.min())).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask)
    x, y, w, h = st[lab[peak], :4]                                     # 최고점이 속한 덩어리의 박스
    corners = np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], np.float32)
    Minv = cv2.invertAffineTransform(it["M"])
    orig = cv2.transform(corners[None], Minv)[0]                       # 원본 좌표의 회전된 사각형
    x0, y0 = np.clip(orig.min(0), 0, [W, H]); x1, y1 = np.clip(orig.max(0), 0, [W, H])
    with open(os.path.join(DRAFT_DIR, "labels", it["file"].replace(".png", ".txt")), "w") as fp:
        fp.write(f"0 {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}\n")
    vis = cv2.imread(it["path"])
    cv2.rectangle(vis, (int(x0), int(y0)), (int(x1), int(y1)), (0, 0, 255), 3)
    cv2.imwrite(os.path.join(DRAFT_DIR, "overlays", it["file"].replace(".png", ".jpg")), vis)
    np.save(os.path.join(DRAFT_DIR, it["file"].replace(".png", "_amap.npy")), amap.astype(np.float16))

print(f"박스 초안 {len(defects)}장 저장: {DRAFT_DIR}")
print("※ 초안은 히트맵 최고점 주변만 잡으므로 결함이 여러 곳이거나 히트맵이 빗나간 경우 반드시 수정 필요")

samples = defects[:6]
fig, axes = plt.subplots(1, len(samples), figsize=(2.6 * len(samples), 3))
for ax, it in zip(axes, samples):
    ax.imshow(cv2.imread(os.path.join(DRAFT_DIR, "overlays", it["file"].replace(".png", ".jpg")))[:, :, ::-1])
    ax.set_title(it["file"][6:-4], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 다음 단계

1. `EXP_NAME`을 `P2_blur_aug`, `P3_light`로 바꿔 다시 실행 → `results_all.csv`에 줄이 쌓임
2. 통과 기준 확인 (보고서 6.1): 평가 AUROC ≥ 0.90, 흐림/선명 AUROC 차이 ≤ 0.10, 히트맵 적중 ≥ 70% (위 히트맵을 육안 검수), 전처리+추론 ≤ 35ms
3. 결과를 `report/model_selection_report.md` 6.4절 표에 기입
4. 통과 시 `yolo_drafts/`의 박스 초안을 라벨링 도구에서 검수 → 2차 YOLO 실험